# Apple Good / Bad Classifier: 5-Fold Cross-Validation Training (Kaggle)

**Setup (once)**
1. Ensure your dataset (e.g. `applezip`) is attached to this notebook in the right panel (**Add Input**).
2. In the right panel settings: **Accelerator = GPU** (T4 x2 or P100), **Internet = On** (needed once to pull pretrained ConvNeXt backbone).
3. Click **Run All** from top to bottom.

**What this notebook does**
- Loads all 415 labeled apples into memory and audits class / variety / lighting distribution.
- Groups multiple photos of the same apple so one apple is never split across training and validation.
- Trains ConvNeXt-Tiny with 5-fold grouped cross-validation, balanced sampling, dynamic background replacement, and photometric lighting augmentation.
- Reports accuracy per variety and lighting, tests against shortcut probes, and visualizes defect attention heatmaps with Grad-CAM.
- Saves model weights in `/kaggle/working/run/models/` and generates `results.zip`.


In [ ]:
# 1. Install (timm is usually preinstalled on Kaggle; this ensures it is present)
!pip install -q timm


In [ ]:
# 2. Imports and CONFIG
import os, re, glob, json, math, random, time, shutil, warnings, copy
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")

INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working/run"

# ------------------------- training settings -------------------------
QUICK_TEST    = False     # False = full 5-fold training and rigorous evaluation
EXCLUDE_SEG_PROBLEMS = True   # drop images flagged with Q (bad cutouts)

MODEL_NAME    = "convnext_tiny"
PRETRAINED    = True
IMG_SIZE      = 384
EPOCHS        = 30
BATCH         = 24
LR_BACKBONE   = 1e-4
LR_HEAD       = 1e-3
WEIGHT_DECAY  = 0.05
WARMUP_EPOCHS = 2
LABEL_SMOOTH  = 0.05
DROP_PATH     = 0.1
EMA_DECAY     = 0.99
N_FOLDS       = 5
SEED          = 42
NUM_WORKERS   = 2
BALANCE_POWER = 1.0       # balance variety x lighting x label cells
MIN_CELL      = 15
APPLE_FILL    = 0.92      # apple size relative to canvas
AUG_STRENGTH  = 1.0       # 1.0 = full background & lighting augmentation
AUTO_EASE_AUG = True      # automatically eases augmentation if loss stays high
AUG_RAMP      = True      # ramp up augmentation gradually during early epochs
RUN_DINO_BASELINE = True  # frozen DINOv2 baseline comparison
SAVE_MODELS   = True

# ------------------------- grouping of photos of the same apple -------------------------
GROUP_GAP_SECONDS = 20
SIM_LINK_PCT  = 90
DUP_PCT       = 99.9
MAX_GROUP     = 8

if QUICK_TEST:
    EPOCHS, IMG_SIZE, N_RUN_FOLDS = 3, 224, 1
else:
    N_RUN_FOLDS = N_FOLDS

AUG_NOW = AUG_STRENGTH
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
PRELOAD_SIDE = 512
os.makedirs(WORK, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)
print("device:", DEVICE, "| torch", torch.__version__, "| timm", timm.__version__)
if QUICK_TEST:
    print("*** QUICK_TEST is ON: smoke test run ***")


In [ ]:
# 3. Load labels and images, compute stats and audit the dataset
cands = sorted(glob.glob(f"{INPUT_ROOT}/**/labels.csv", recursive=True))
assert cands, "No labels.csv found under /kaggle/input. Attach your dataset in the right panel (Add Input)."
def nrows(p):
    try: return len(pd.read_csv(p))
    except Exception: return 0
LABELS_CSV = max(cands, key=nrows)
print("labels file:", LABELS_CSV, "| found", len(cands), "labels.csv file(s)")
lab = pd.read_csv(LABELS_CSV)
lab.columns = [c.strip().lower() for c in lab.columns]
assert {"filename", "label"} <= set(lab.columns), f"labels.csv needs columns filename,label. It has: {list(lab.columns)}"
lab["stem"] = lab["filename"].astype(str).map(lambda s: Path(s).stem)
lab["label"] = lab["label"].astype(str).str.strip().str.lower()
print("label counts in the file:", lab["label"].value_counts(dropna=False).to_dict())
lab = lab[lab["label"].isin(["good", "bad"])].drop_duplicates("stem").copy()

# --- find the image files (RGBA cutouts and gray JPGs), matched by filename stem ---
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rgba_idx, gray_idx = {}, {}
for p in glob.glob(f"{INPUT_ROOT}/**/*", recursive=True):
    ext = Path(p).suffix.lower()
    if ext not in IMG_EXT: continue
    stem = Path(p).stem
    if ext == ".png":
        try:
            if Image.open(p).mode == "RGBA":
                rgba_idx.setdefault(stem, p); continue
        except Exception:
            pass
    gray_idx.setdefault(stem, p)
lab["rgba"] = lab["stem"].map(rgba_idx)
lab["gray"] = lab["stem"].map(gray_idx)
lab["path"] = lab["rgba"].fillna(lab["gray"])
miss = lab["path"].isna()
if miss.any():
    print(f"WARNING: {int(miss.sum())} labeled images have no image file and are dropped, e.g.", lab.loc[miss, "stem"].head(5).tolist())
lab = lab[~miss].copy()
print(f"images with RGBA cutout: {int(lab['rgba'].notna().sum())} | gray-only (mask recovered automatically): {int(lab['rgba'].isna().sum())}")

if "seg_problem" in lab.columns:
    lab["seg_problem"] = pd.to_numeric(lab["seg_problem"], errors="coerce").fillna(0).astype(int)
    if EXCLUDE_SEG_PROBLEMS:
        n_seg = int(lab["seg_problem"].sum())
        if n_seg > 0:
            print("dropping images flagged as segmentation problems:", n_seg)
            lab = lab[lab["seg_problem"] == 0].copy()

def norm_variety(v):
    v = str(v).strip().lower()
    if v == "red": return "red"
    if v in ("green", "yellow", "green/yellow", "green_yellow", "green-yellow"): return "green_yellow"
    return "unknown"
lab["variety"] = lab["variety"].map(norm_variety) if "variety" in lab.columns else "unknown"

# --- load everything into memory (apple RGB + alpha) ---
CACHE = {}
def load_rgb_alpha(path):
    im = Image.open(path)
    if im.mode == "RGBA":
        a = np.array(im)
        return a[..., :3].copy(), a[..., 3].copy()
    rgb = np.array(im.convert("RGB"))
    # gray-only image: recover apple mask from neutral gray (128) background
    m = (np.abs(rgb.astype(np.int16) - 128).max(2) > 10).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    l, n = ndi.label(m)
    if n:
        sizes = ndi.sum(m, l, range(1, n + 1))
        m = ndi.binary_fill_holes(l == (int(np.argmax(sizes)) + 1)).astype(np.uint8)
    return rgb, m * 255

def prep(rgb, a):
    s = min(1.0, PRELOAD_SIDE / max(a.shape))
    if s < 1.0:
        size = (max(1, round(a.shape[1] * s)), max(1, round(a.shape[0] * s)))
        rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
        a = cv2.resize(a, size, interpolation=cv2.INTER_AREA)
    return rgb, a

def apple_features(rgb, a):
    s = 384 / max(a.shape)
    size = (max(1, round(a.shape[1] * s)), max(1, round(a.shape[0] * s)))
    rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA); a = cv2.resize(a, size, interpolation=cv2.INTER_AREA)
    m = a > 127
    if m.sum() < 50:
        return dict(f_L=np.nan, f_a=np.nan, f_b=np.nan, f_sat=np.nan, f_sharp=np.nan, f_area=0.0)
    L = cv2.cvtColor(rgb.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    er = cv2.erode(m.astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    sharp = float(lap[er].var()) if er.sum() > 50 else float(lap[m].var())
    return dict(f_L=float(L[..., 0][m].mean()), f_a=float(L[..., 1][m].mean()), f_b=float(L[..., 2][m].mean()),
                f_sat=float(hsv[..., 1][m].mean()), f_sharp=float(np.log1p(sharp)), f_area=float(m.mean()))

feats = []
for stem, p in zip(lab["stem"], lab["path"]):
    rgb, a = load_rgb_alpha(p)
    feats.append(apple_features(rgb, a))
    CACHE[stem] = prep(rgb, a)
lab = pd.concat([lab.reset_index(drop=True), pd.DataFrame(feats)], axis=1)
lab = lab[lab["f_area"] > 0].copy()

from skimage.filters import threshold_otsu
L_THR = float(threshold_otsu(lab["f_L"].values))
lab["lighting_auto"] = np.where(lab["f_L"] < L_THR, "dim", "normal")
if "lighting" in lab.columns:
    lg = lab["lighting"].astype(str).str.strip().str.lower()
    lab["lighting"] = np.where(lg.isin(["normal", "dim"]), lg, lab["lighting_auto"])
else:
    lab["lighting"] = lab["lighting_auto"]

def parse_ts(name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})[_\-]?(\d{2})(\d{2})(\d{2})", name)
    if m:
        try: return pd.Timestamp(*map(int, m.groups()))
        except Exception: return pd.NaT
    m = re.search(r"@(\d{2})-(\d{2})-(\d{4})_(\d{2})-(\d{2})-(\d{2})", name)
    if m:
        d, mo, y, h, mi, s = map(int, m.groups())
        try: return pd.Timestamp(y, mo, d, h, mi, s)
        except Exception: return pd.NaT
    return pd.NaT

lab["ts"] = lab["stem"].map(parse_ts)

def show(title, df):
    print("\n" + title); print(df.to_string())
show("Total Dataset Audit: variety x label", pd.crosstab(lab.variety, lab.label, margins=True))
show("Total Dataset Audit: lighting x label", pd.crosstab(lab.lighting, lab.label, margins=True))
show("Total Dataset Audit: variety x lighting x label", pd.crosstab([lab.variety, lab.lighting], lab.label, margins=True))


In [ ]:
# 4. Augmentation & Dataset
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def place(rgb, a, S, scale=1.0, angle=0.0, flip_h=False, flip_v=False, shift=(0.0, 0.0)):
    if flip_h: rgb, a = rgb[:, ::-1], a[:, ::-1]
    if flip_v: rgb, a = rgb[::-1], a[::-1]
    h, w = a.shape
    T = APPLE_FILL * S * scale
    k = T / max(h, w)
    size = (max(2, round(w * k)), max(2, round(h * k)))
    interp = cv2.INTER_AREA if k < 1 else cv2.INTER_LINEAR
    rgb = cv2.resize(np.ascontiguousarray(rgb), size, interpolation=interp)
    a = cv2.resize(np.ascontiguousarray(a), size, interpolation=interp)
    h, w = a.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[0, 2] += S / 2 - w / 2 + shift[0] * S
    M[1, 2] += S / 2 - h / 2 + shift[1] * S
    rgb_o = cv2.warpAffine(rgb, M, (S, S), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    a_o = cv2.warpAffine(a, M, (S, S), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return rgb_o.astype(np.float32), (a_o.astype(np.float32) / 255.0)

def composite(rgb, alpha, bg):
    return rgb * alpha[..., None] + bg * (1 - alpha[..., None])

def aug_eval(rgb, a, S):
    r, al = place(rgb, a, S)
    return np.clip(composite(r, al, np.full((S, S, 3), 128, np.float32)), 0, 255).astype(np.uint8)

def make_bg(rng, S):
    p_gray = 1 - 0.55 * AUG_NOW
    r = rng.random()
    if r < p_gray: return np.full((S, S, 3), 128, np.float32)
    r = (r - p_gray) / max(1e-6, 1 - p_gray)
    if r < 0.27: return np.full((S, S, 3), float(rng.integers(50, 210)), np.float32)
    if r < 0.64: return np.tile(rng.integers(20, 235, 3).astype(np.float32), (S, S, 1))
    low = rng.random((6, 6, 3)).astype(np.float32) * 255
    bg = cv2.resize(low, (S, S), interpolation=cv2.INTER_CUBIC)
    return np.clip(bg + rng.normal(0, 6, bg.shape).astype(np.float32), 0, 255)

def photometric(rgb, alpha, rng):
    A = AUG_NOW
    x = np.clip(rgb / 255.0, 0, 1)
    x = x ** math.exp(rng.uniform(math.log(0.7), math.log(1.45)) * A)
    w = alpha[..., None]
    mean = (x * w).sum() / (w.sum() * 3 + 1e-6)
    x = (x - mean) * (1 + (rng.uniform(0.75, 1.25) - 1) * A) + mean
    x = x * math.exp(rng.uniform(math.log(0.55), math.log(1.5)) * A)
    t = rng.normal(0, 0.05) * A
    x[..., 0] *= 1 + t; x[..., 2] *= 1 - t
    g = x.mean(2, keepdims=True)
    x = g + (x - g) * (1 + (rng.uniform(0.8, 1.2) - 1) * A)
    x = x * rng.uniform(0.96, 1.04, 3).astype(np.float32)
    return np.clip(x, 0, 1) * 255.0

def motion_blur(img, rng):
    n = int(rng.integers(5, 16))
    k = np.zeros((n, n), np.float32); k[n // 2, :] = 1
    M = cv2.getRotationMatrix2D((n / 2 - 0.5, n / 2 - 0.5), rng.uniform(0, 180), 1)
    k = cv2.warpAffine(k, M, (n, n)); k /= max(k.sum(), 1e-6)
    return cv2.filter2D(img, -1, k)

def aug_train(rgb, a, S, rng):
    scale = rng.uniform(0.85, 1.08)
    angle = rng.uniform(-45, 45) if rng.random() < 0.7 else rng.uniform(-180, 180)
    r, al = place(rgb, a, S, scale=scale, angle=angle, flip_h=rng.random() < 0.5, flip_v=rng.random() < 0.5,
                  shift=(rng.uniform(-0.04, 0.04), rng.uniform(-0.04, 0.04)))
    r = photometric(r, al, rng)
    img = composite(r, al, make_bg(rng, S))
    if rng.random() < 0.35 * AUG_NOW: img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, 2.0))
    if rng.random() < 0.15 * AUG_NOW: img = motion_blur(img, rng)
    if rng.random() < 0.30 * AUG_NOW: img = img + rng.normal(0, rng.uniform(2, 8), img.shape).astype(np.float32)
    img = np.clip(img, 0, 255).astype(np.uint8)
    if rng.random() < 0.30 * AUG_NOW:
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(40, 90))])
        if ok: img = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return img

class AppleDS(Dataset):
    def __init__(self, df, train, size=None, salt=0):
        self.stems = df["stem"].tolist()
        self.y = (df["label"] == "bad").astype(int).tolist() if "label" in df.columns else [0] * len(df)
        self.train, self.S, self.salt, self.epoch, self.calls = train, size or IMG_SIZE, salt, 0, 0
    def __len__(self): return len(self.stems)
    def __getitem__(self, i):
        rgb, a = CACHE[self.stems[i]]
        if self.train:
            self.calls += 1
            rng = np.random.default_rng([SEED, self.salt, self.epoch, i, self.calls])
            img = aug_train(rgb, a, self.S, rng)
        else:
            img = aug_eval(rgb, a, self.S)
        x = (torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1) - MEAN) / STD
        return x, self.y[i]

# preview what the model sees during training vs evaluation
prev = lab.sample(min(12, len(lab)), random_state=1)
ds_prev = AppleDS(prev, True, 224)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax in axes.ravel(): ax.axis("off")
for j in range(min(12, len(prev))):
    row = prev.iloc[j]
    r_, c_ = j // 4, (j % 4) * 2
    axes[r_][c_].imshow(aug_eval(*CACHE[row.stem], 224)); axes[r_][c_].set_title(f"{row.label} (eval view)", fontsize=8)
    x, _ = ds_prev[j]
    axes[r_][c_ + 1].imshow(((x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy())); axes[r_][c_ + 1].set_title("train view", fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/augmentation_preview.jpg", dpi=60); plt.show()


In [ ]:
# 5. Group photos of the same apple
def make_model(pretrained=None, num_classes=2):
    pretrained = PRETRAINED if pretrained is None else pretrained
    return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes, drop_path_rate=DROP_PATH)

@torch.no_grad()
def embed_df(df, size=None):
    m = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=0).to(DEVICE).eval()
    dl = DataLoader(AppleDS(df, False, size), batch_size=32, shuffle=False, num_workers=NUM_WORKERS)
    out = []
    for x, _ in dl:
        f = m(x.to(DEVICE)).float()
        out.append(F.normalize(f, dim=1).cpu().numpy())
    del m
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return np.concatenate(out)

lab = lab.sort_values(["ts", "stem"], na_position="last").reset_index(drop=True)
E = embed_df(lab)
SIM = E @ E.T
N = len(lab)
cls = np.asarray((lab.label + "|" + lab.variety).tolist(), dtype=object)
same_cls = cls[:, None] == cls[None, :]
iu = np.triu_indices(N, 1)
rand_sims = SIM[iu][same_cls[iu]]
THR_LINK = float(np.percentile(rand_sims, SIM_LINK_PCT))
THR_DUP = float(np.percentile(rand_sims, DUP_PCT))
print(f"similarity thresholds: link >= {THR_LINK:.3f} | near-duplicate >= {THR_DUP:.3f}")

cand = []
for i in range(1, N):
    if pd.notna(lab.at[i, "ts"]) and pd.notna(lab.at[i - 1, "ts"]) and same_cls[i - 1, i]:
        if (lab.at[i, "ts"] - lab.at[i - 1, "ts"]).total_seconds() <= GROUP_GAP_SECONDS and SIM[i - 1, i] >= THR_LINK:
            cand.append((float(SIM[i - 1, i]), i - 1, i, "time"))
ii, jj = np.where(np.triu(same_cls & (SIM >= THR_DUP), 1))
for i, j in zip(ii, jj):
    cand.append((float(SIM[i, j]), int(i), int(j), "dup"))
cand.sort(reverse=True)

parent = list(range(N)); size = [1] * N
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i

n_t = n_d = n_refused = 0
for s, i, j, kind in cand:
    ri, rj = find(i), find(j)
    if ri == rj: continue
    if size[ri] + size[rj] > MAX_GROUP:
        n_refused += 1; continue
    parent[rj] = ri; size[ri] += size[rj]
    if kind == "time": n_t += 1
    else: n_d += 1

lab["group"] = [find(i) for i in range(N)]
gs = lab.groupby("group").size()
print(f"links made: {n_t} by time+look, {n_d} near-duplicate | refused: {n_refused}")
print(f"{N} photos grouped into {len(gs)} unique apple groups.")


In [ ]:
# 6. Folds (grouped + stratified) across all photos
def assign_folds(df):
    key = df["label"] + "_" + df["variety"]
    fold = np.full(len(df), -1)
    if df["group"].nunique() < N_FOLDS:
        return df["group"].astype("category").cat.codes.values % N_FOLDS
    sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(sg.split(df, key, df["group"])): fold[va] = k
    return fold

lab["fold"] = assign_folds(lab)
show("Photos per fold (label_variety)", pd.crosstab(lab.fold, lab.label + "_" + lab.variety))

# --- leakage audit: validation photos vs training photos ---
pairs = []
oi = lab.index.values
for k in range(N_FOLDS):
    va = oi[lab.loc[oi, "fold"].values == k]; tr = oi[lab.loc[oi, "fold"].values != k]
    if len(va) == 0: continue
    sub = SIM[np.ix_(va, tr)]
    j = sub.argmax(1)
    for a, b, s in zip(va, tr[j], sub.max(1)): pairs.append((float(s), int(a), int(b)))
pairs.sort(reverse=True)
n_hi = sum(1 for s, _, _ in pairs if s >= THR_DUP)
print(f"validation photos whose closest training photo is a near-duplicate: {n_hi} of {len(pairs)} (ideal = 0)")

# --- shortcut probes ---
PROBES = {
    "variety only": ["v_red", "v_gy"],
    "capture stats only (brightness, colour, sharpness)": ["f_L", "f_a", "f_b", "f_sat", "f_sharp"],
    "variety + capture stats": ["v_red", "v_gy", "f_L", "f_a", "f_b", "f_sat", "f_sharp"],
}
lab["v_red"] = (lab.variety == "red").astype(float); lab["v_gy"] = (lab.variety == "green_yellow").astype(float)
def probe_oof(df, cols):
    oof = np.full(len(df), np.nan)
    X, y, fo = df[cols].values, (df.label == "bad").astype(int).values, df.fold.values
    for k in range(N_FOLDS):
        tr, va = fo != k, fo == k
        if va.sum() == 0 or len(set(y[tr])) < 2: continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced"))
        clf.fit(X[tr], y[tr]); oof[va] = clf.predict_proba(X[va])[:, 1]
    return oof
def auc_safe(y, p):
    ok = ~np.isnan(p)
    return roc_auc_score(y[ok], p[ok]) if len(set(y[ok])) == 2 else np.nan

PROBE_AUC = {}
y_all = (lab.label == "bad").astype(int).values
rows = []
for name, cols in PROBES.items():
    o = probe_oof(lab, cols)
    PROBE_AUC[name] = auc_safe(y_all, o)
    row = {"probe": name, "AUC all": PROBE_AUC[name]}
    for v in ("red", "green_yellow"):
        mk = (lab.variety == v).values
        row[f"AUC within {v}"] = auc_safe(y_all[mk], o[mk])
    rows.append(row)
print("\nSHORTCUT PROBES (AUC 0.5 = random, 1.0 = perfect):")
print(pd.DataFrame(rows).round(3).to_string(index=False))


In [ ]:
# 7. Training machinery
class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval(); self.decay = decay
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for e, p in zip(self.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
            else: e.copy_(p)

def cell_weights(df):
    key = df["label"] + "_" + df["variety"] + "_" + df["lighting"]
    cnt = key.map(key.value_counts()).values.astype(float)
    w = 1.0 / np.power(np.maximum(cnt, MIN_CELL), BALANCE_POWER)
    return w / w.sum() * len(df)

@torch.no_grad()
def predict(model, df, tta=True, size=None):
    model.eval()
    dl = DataLoader(AppleDS(df, False, size), batch_size=32, shuffle=False, num_workers=NUM_WORKERS)
    out = []
    for x, _ in dl:
        x = x.to(DEVICE).contiguous(memory_format=torch.channels_last)
        views = [x] + ([x.flip(3), x.flip(2), x.flip(2).flip(3)] if tta else [])
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            p = torch.stack([F.softmax(model(v).float(), 1)[:, 1] for v in views]).mean(0)
        out.append(p.cpu().numpy())
    return np.concatenate(out)

STATES = {}
LAST_TRAIN_LOSS = 0.0
def run_fold(train_df, val_df, exp, fold):
    global AUG_NOW, LAST_TRAIN_LOSS
    t0 = time.time()
    seed_everything(SEED + fold)
    model = make_model().to(DEVICE).to(memory_format=torch.channels_last)
    ema = EMA(model, EMA_DECAY)
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    body = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": body, "lr": LR_BACKBONE}, {"params": head, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    base_lrs = [LR_BACKBONE, LR_HEAD]
    bs = max(2, min(BATCH, len(train_df) // 2))
    ds = AppleDS(train_df, True, salt=fold)
    sampler = WeightedRandomSampler(torch.tensor(cell_weights(train_df)), num_samples=len(train_df), replacement=True,
                                    generator=torch.Generator().manual_seed(SEED + fold))
    dl = DataLoader(ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True, pin_memory=(DEVICE == "cuda"))
    total, warm = EPOCHS * len(dl), WARMUP_EPOCHS * len(dl)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    step = 0
    for ep in range(EPOCHS):
        ds.epoch = ep; model.train(); run = 0.0; nb = 0
        AUG_NOW = AUG_STRENGTH * (min(1.0, 0.3 + 0.7 * ep / max(1.0, 0.6 * EPOCHS)) if AUG_RAMP else 1.0)
        for x, y in dl:
            f = (step + 1) / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * f
            x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last); y = y.to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                loss = crit(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); ema.update(model)
            step += 1; run += float(loss); nb += 1
        msg = f"  [fold {fold}] epoch {ep + 1}/{EPOCHS} loss {run / max(1, nb):.3f}"
        if (ep + 1) % 5 == 0 or ep + 1 == EPOCHS:
            pv = predict(ema.m, val_df, tta=False)
            yv = (val_df.label == "bad").astype(int).values
            msg += f" | val balanced acc {balanced_accuracy_score(yv, pv >= 0.5):.3f}"
        print(msg, flush=True)
    AUG_NOW = AUG_STRENGTH
    LAST_TRAIN_LOSS = run / max(1, nb)
    probs = predict(ema.m, val_df, tta=True)
    state = {k: (v.half() if v.dtype.is_floating_point else v).cpu() for k, v in ema.m.state_dict().items()}
    STATES[fold] = state
    if SAVE_MODELS:
        os.makedirs(f"{WORK}/models", exist_ok=True)
        torch.save(state, f"{WORK}/models/convnext_fold{fold}.pt")
    print(f"  fold {fold} done in {(time.time() - t0) / 60:.1f} min", flush=True)
    del model, ema, opt
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return probs


In [ ]:
# 8. Train ConvNeXt-Tiny across 5 folds on the complete dataset
print(f"\n===== Training ConvNeXt-Tiny across {N_RUN_FOLDS} folds on all {len(lab)} apples =====")
oof = pd.Series(np.nan, index=lab.index)
EASED_ONCE = False
for k in range(N_RUN_FOLDS):
    tr = lab[lab.fold != k]; va = lab[lab.fold == k]
    if len(va) == 0:
        print(f"  fold {k} is empty, skipped"); continue
    probs = run_fold(tr, va, "convnext", k)
    if AUTO_EASE_AUG and not QUICK_TEST and not EASED_ONCE:
        EASED_ONCE = True
        tries = 0
        while LAST_TRAIN_LOSS > 0.55 and AUG_STRENGTH > 0.2 and tries < 2:
            AUG_STRENGTH = AUG_STRENGTH * 0.5; tries += 1
            print(f"  -> Lowering AUG_STRENGTH to {AUG_STRENGTH} and re-running fold {k}.")
            probs = run_fold(tr, va, "convnext", k)
    oof.loc[va.index] = probs
print(f"\nAll {N_RUN_FOLDS} folds finished successfully! (Augmentation strength used: {AUG_STRENGTH})")


In [ ]:
# 9. Results: Complete Honest Evaluation
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def summarize(df, p, thr=0.5):
    d = df.copy(); d["p_bad"] = p; d["pred"] = np.where(p >= thr, "bad", "good"); d["ok"] = (d.pred == d.label)
    y = (d.label == "bad").astype(int).values
    has_both = len(set(y)) == 2
    out = dict(n=len(d), accuracy=d.ok.mean(),
               balanced_acc=balanced_accuracy_score(y, p >= thr) if has_both else d.ok.mean(),
               AUC=roc_auc_score(y, p) if has_both else np.nan,
               recall_good=d[d.label == "good"].ok.mean() if (d.label == "good").any() else np.nan,
               recall_bad=d[d.label == "bad"].ok.mean() if (d.label == "bad").any() else np.nan)
    return d, out

def cell_table(d, by):
    rows = []
    for key, g in d.groupby(by):
        lo, hi = wilson(int(g.ok.sum()), len(g))
        rows.append({**dict(zip(by, key if isinstance(key, tuple) else (key,))), "n": len(g), "correct": f"{g.ok.mean():.0%}", "95% range": f"{lo:.0%}-{hi:.0%}"})
    return pd.DataFrame(rows)

d, s = summarize(lab, oof.values)

print(f"\n=================== OVERALL 5-FOLD EVALUATION ({len(d)} apples) ===================")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in s.items()})
print("\nAccuracy per variety x label:");              print(cell_table(d, ["variety", "label"]).to_string(index=False))
print("\nAccuracy per variety x lighting x label:");    print(cell_table(d, ["variety", "lighting", "label"]).to_string(index=False))

print("\nSHORTCUT PROBE AUCs:", {k: round(v, 3) for k, v in PROBE_AUC.items()})
y = (d.label == "bad").astype(int).values
print(f"Model AUC: {roc_auc_score(y, d.p_bad):.3f} vs best shortcut probe: {max(PROBE_AUC.values()):.3f}")

# Threshold trade-off
rows = []
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    bad, good = d[d.label == "bad"], d[d.label == "good"]
    rows.append({"threshold": t, "bad missed (called good)": f"{(bad.p_bad < t).mean():.0%}", "good rejected (called bad)": f"{(good.p_bad >= t).mean():.0%}"})
print(f"\nTHRESHOLD TRADE-OFF: lower threshold = fewer bad apples slip through")
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for lbl, c in (("good", "tab:green"), ("bad", "tab:red")):
    ax[0].hist(d[d.label == lbl].p_bad, bins=20, range=(0, 1), alpha=0.6, label=lbl, color=c)
ax[0].set_title("Predicted P(bad)"); ax[0].legend()
cm = pd.crosstab(d.label, d.pred).reindex(index=["good", "bad"], columns=["good", "bad"]).fillna(0).values
ax[1].imshow(cm, cmap="Blues"); ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["pred good", "pred bad"]); ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["true good", "true bad"])
for i in range(2):
    for j in range(2): ax[1].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=14)
plt.tight_layout(); plt.savefig(f"{WORK}/results_overview.jpg", dpi=70); plt.show()


In [ ]:
# 10. Frozen DINOv2 baseline (reference sanity check)
if RUN_DINO_BASELINE:
    try:
        dm = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224).to(DEVICE).eval()
        dlb = DataLoader(AppleDS(lab, False, 224), batch_size=32, shuffle=False, num_workers=NUM_WORKERS)
        zs = []
        with torch.no_grad():
            for x, _ in dlb: zs.append(F.normalize(dm(x.to(DEVICE)).float(), dim=1).cpu().numpy())
        Z = np.concatenate(zs); yb = (lab.label == "bad").astype(int).values; fo = lab.fold.values
        oofb = np.full(len(lab), np.nan)
        for k in range(N_FOLDS):
            tr, va = fo != k, fo == k
            if va.sum() == 0: continue
            clf = LogisticRegression(C=10, max_iter=3000, class_weight="balanced").fit(Z[tr], yb[tr])
            oofb[va] = clf.predict_proba(Z[va])[:, 1]
        okb = ~np.isnan(oofb)
        db, sb = summarize(lab[okb], oofb[okb])
        print("\n=========== DINOv2 frozen features + logistic regression baseline ===========")
        print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in sb.items()})
        print(cell_table(db, ["variety", "label"]).to_string(index=False))
        del dm
        if DEVICE == "cuda": torch.cuda.empty_cache()
    except Exception as e:
        print("DINOv2 baseline skipped:", e)


In [ ]:
# 11. Inspect highest-loss images & Grad-CAM defect attention
d["loss"] = -np.log(np.where(d.label == "bad", d.p_bad, 1 - d.p_bad).clip(1e-6, 1))
err = d.sort_values("loss", ascending=False).head(24)
cols = 6; rws = max(1, math.ceil(len(err) / cols))
fig, axes = plt.subplots(rws, cols, figsize=(2.6 * cols, 3.0 * rws))
for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), err.iterrows()):
    ax.imshow(aug_eval(*CACHE[r.stem], 220))
    ax.set_title(f"{r.stem[-14:]}\nlabel {r.label} | P(bad) {r.p_bad:.2f}\n{r.variety} {r.lighting}", fontsize=7, color="red" if r.pred != r.label else "black")
plt.suptitle("Highest-loss images: check if any label was mistyped in the tool", fontsize=10)
plt.tight_layout(); plt.savefig(f"{WORK}/errors_to_review.jpg", dpi=65); plt.show()
err[["stem", "label", "p_bad", "variety", "lighting"]].to_csv(f"{WORK}/errors_to_review.csv", index=False)

def gradcam(model, x, cls):
    stage = model.stages[-1]
    store = {}
    def fh(m, i, o): o.retain_grad(); store["o"] = o
    h = stage.register_forward_hook(fh)
    model.zero_grad(); logits = model(x); logits[0, cls].backward(); h.remove()
    o = store["o"]; w = o.grad.mean((2, 3), keepdim=True)
    cam = F.relu((w * o).sum(1, keepdim=True)); cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy()

try:
    model = make_model(pretrained=False).to(DEVICE)
    model.load_state_dict({k: (v.float() if v.dtype == torch.float16 else v) for k, v in STATES[0].items()})
    model.eval()
    dv = d[lab.loc[d.index, "fold"] == 0]
    pick = pd.concat([dv[(dv.label == "bad") & (dv.pred == "bad")].head(4), dv[dv.pred != dv.label].head(4), dv[(dv.label == "good") & (dv.pred == "good")].head(2)])
    if len(pick) > 0:
        fig, axes = plt.subplots(2, len(pick), figsize=(2.6 * len(pick), 5.4))
        axes = np.array(axes).reshape(2, -1)
        for c, (_, r) in enumerate(pick.iterrows()):
            img = aug_eval(*CACHE[r.stem], IMG_SIZE)
            x = ((torch.from_numpy(img.astype(np.float32) / 255).permute(2, 0, 1) - MEAN) / STD)[None].to(DEVICE)
            cam = gradcam(model, x, 1 if r.pred == "bad" else 0)
            heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
            axes[0][c].imshow(img); axes[0][c].axis("off"); axes[0][c].set_title(f"label {r.label} / pred {r.pred}", fontsize=7)
            axes[1][c].imshow((0.55 * img + 0.45 * heat).astype(np.uint8)); axes[1][c].axis("off")
        plt.suptitle("Grad-CAM: red = where the model attends (should focus on defects)", fontsize=9)
        plt.tight_layout(); plt.savefig(f"{WORK}/gradcam.jpg", dpi=65); plt.show()
    del model
except Exception as e:
    print("Grad-CAM skipped:", e)


In [ ]:
# 12. Save predictions and package results
out = lab[["stem", "label", "variety", "lighting", "group", "fold"]].copy()
out["p_bad"] = oof
out.to_csv(f"{WORK}/oof_predictions.csv", index=False)
json.dump({"config": dict(MODEL_NAME=MODEL_NAME, IMG_SIZE=IMG_SIZE, EPOCHS=EPOCHS, QUICK_TEST=QUICK_TEST, N_FOLDS=N_FOLDS, SEED=SEED),
           "probe_auc": {k: float(v) for k, v in PROBE_AUC.items()}}, open(f"{WORK}/summary.json", "w"), indent=1)

tmpdir = "/kaggle/working/results_small"; shutil.rmtree(tmpdir, ignore_errors=True)
shutil.copytree(WORK, tmpdir, ignore=shutil.ignore_patterns("models"))
shutil.make_archive("/kaggle/working/results", "zip", tmpdir)
print("results.zip (metrics & plots):", round(os.path.getsize("/kaggle/working/results.zip") / 1e6, 1), "MB")
if os.path.isdir(f"{WORK}/models"):
    print("Model checkpoints:", round(sum(os.path.getsize(f) for f in glob.glob(f"{WORK}/models/*.pt")) / 1e6), "MB in", f"{WORK}/models")
